# ⚔️ 13. Grayscale NTSC Training SVM & Injeksi Final (.keras)

Notebook ini melatih Support Vector Machine (SVM) dengan RBF Kernel pada vektor fitur CNN hasil ekstraksi Tahap 12.
- **Anti Data Leakage**: StandardScaler hanya di-it pada data training.
- **Injeksi Final .keras**: Model dasar CNN, StandardScaler, dan model SVM Classifier disimpan bersama ke dalam **SATU file tunggal** custom_cnn_cifar10_final.keras via format HDF5.


In [1]:
import os
import sys
import h5py
import pickle
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

ARTIFACT_DIR = 'cifar10_artifacts/grayscale_ntsc'
train_feat_path = os.path.join(ARTIFACT_DIR, 'train_features.npz')
print(f"Memuat vektor fitur training dari: {train_feat_path}")
train_data = np.load(train_feat_path)
X_train_features = train_data['X_train_features']
y_train = train_data['y_train']

RANDOM_STATE = 23092026

def save_and_inject_final_model(cnn_model, scaler, svm_model, artifact_path, metadata=None):
    os.makedirs(os.path.dirname(artifact_path), exist_ok=True)
    cnn_model.save(artifact_path)
    scaler_bytes = pickle.dumps(scaler, protocol=pickle.HIGHEST_PROTOCOL)
    svm_bytes = pickle.dumps(svm_model, protocol=pickle.HIGHEST_PROTOCOL)
    with h5py.File(artifact_path, 'a') as h5f:
        if 'scaler_pickle' in h5f:
            del h5f['scaler_pickle']
        if 'svm_model_pickle' in h5f:
            del h5f['svm_model_pickle']
        if 'svm_pipeline_pickle' in h5f:
            del h5f['svm_pipeline_pickle']
        h5f.create_dataset('scaler_pickle', data=np.frombuffer(scaler_bytes, dtype=np.uint8), compression='gzip')
        h5f.create_dataset('svm_model_pickle', data=np.frombuffer(svm_bytes, dtype=np.uint8), compression='gzip')
        if metadata:
            for k, v in metadata.items():
                if isinstance(v, (int, float, str)):
                    h5f.attrs[k] = v
                else:
                    h5f.attrs[k] = str(v)
    print(f"[BERHASIL] Seluruh model (CNN + Scaler + SVM) tersimpan dalam SATU file (.keras): {artifact_path}")

def load_injected_final_model(artifact_path):
    if not os.path.exists(artifact_path):
        raise FileNotFoundError(f"File model {artifact_path} tidak ditemukan!")
    cnn_model = tf.keras.models.load_model(artifact_path)
    with h5py.File(artifact_path, 'r') as h5f:
        if 'scaler_pickle' in h5f and 'svm_model_pickle' in h5f:
            scaler_bytes = bytes(h5f['scaler_pickle'][()])
            svm_bytes = bytes(h5f['svm_model_pickle'][()])
            scaler = pickle.loads(scaler_bytes)
            svm_model = pickle.loads(svm_bytes)
        elif 'svm_pipeline_pickle' in h5f:
            pipe_bytes = bytes(h5f['svm_pipeline_pickle'][()])
            pipe_obj = pickle.loads(pipe_bytes)
            if hasattr(pipe_obj, 'named_steps'):
                scaler = pipe_obj.named_steps.get('scaler', None)
                svm_model = pipe_obj.named_steps.get('svm', pipe_obj)
            else:
                scaler = None
                svm_model = pipe_obj
        else:
            raise KeyError(f"Dataset Scaler/SVM tidak ditemukan dalam {artifact_path}!")
        metadata = {}
        for k, v in h5f.attrs.items():
            if isinstance(v, bytes):
                metadata[k] = v.decode('utf-8')
            else:
                metadata[k] = v
    return cnn_model, scaler, svm_model, metadata


Memuat vektor fitur training dari: cifar10_artifacts/grayscale_ntsc\train_features.npz


## 🛠️ 1. Standarisasi Fitur & Melatih Model SVM Classifier (Anti Data Leakage)


In [2]:
# 1. Standarisasi Fitur Training (Fit & Transform pada data training saja - Anti Data Leakage)
scaler = StandardScaler()
print("Menstandarisasi vektor fitur training dengan StandardScaler...")
X_train_scaled = scaler.fit_transform(X_train_features)
print(f"[BERHASIL] Fitur training telah distandarisasi. Shape: {X_train_scaled.shape}")

# 2. Hyperparameter Support Vector Machine (Configurable)
SVM_C = 10.0           # Default C (pilihan: 1, 3, 5, 8, 10, 15, 20, 25, 30)
SVM_GAMMA = 'scale'    # Default gamma ('scale' atau multiplier: 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)
KERNEL = 'rbf'

# Opsi Validation-Driven Grid Search Tuning (Disjoint Validation Set dari CNN)
# Test set 10.000 terisolasi murni dan TIDAK BOLEH disentuh dalam proses ini
ENABLE_VAL_TUNING = False  # Ubah ke True untuk menjalankan tuning otomatis

C_CANDIDATES = [1.0, 3.0, 5.0, 8.0, 10.0, 15.0, 20.0, 25.0, 30.0]
GAMMA_MULTIPLIERS = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]

if ENABLE_VAL_TUNING:
    print("\n" + "=" * 68)
    print("  VALIDATION-DRIVEN HYPERPARAMETER SEARCH (DISJOINT DARI CNN VALIDATION)")
    print("=" * 68)
    
    from sklearn.model_selection import train_test_split
    
    # 1. Identifikasi 5.000 sampel validation yang digunakan CNN (stratified 10% dengan RANDOM_STATE)
    indices_all = np.arange(len(X_train_features))
    idx_cnn_train, idx_cnn_val = train_test_split(
        indices_all,
        test_size=0.10,
        random_state=RANDOM_STATE,
        stratify=y_train
    )
    
    # 2. Partisi validation SVM (5.000 sampel) diambil MURNI dari data training CNN (45.000 sampel).
    # Hasilnya: validation CNN dan validation SVM 100% DISJOINT (zero overlap / tidak ada sampel yang sama).
    idx_svm_subtrain, idx_svm_val = train_test_split(
        idx_cnn_train,
        test_size=5000,
        random_state=RANDOM_STATE + 100,
        stratify=y_train[idx_cnn_train]
    )
    
    X_tr_sub, y_tr_sub = X_train_features[idx_svm_subtrain], y_train[idx_svm_subtrain]  # 40.000 sampel
    X_va_sub, y_va_sub = X_train_features[idx_svm_val], y_train[idx_svm_val]            # 5.000 sampel disjoint
    
    print(f"Ukuran Sub-Train SVM        : {X_tr_sub.shape} (40.000 sampel, subset training CNN)")
    print(f"Ukuran Disjoint Val SVM    : {X_va_sub.shape} (5.000 sampel, zero overlap dengan CNN val)\n")
    
    # Fit scaler HANYA pada sub-training (anti data leakage terhadap validation split)
    sub_scaler = StandardScaler()
    X_tr_sub_scaled = sub_scaler.fit_transform(X_tr_sub)
    X_va_sub_scaled = sub_scaler.transform(X_va_sub)
    
    gamma_base = 1.0 / (X_tr_sub_scaled.shape[1] * X_tr_sub_scaled.var())
    print(f"Nilai dasar gamma ('scale') pada data training: {gamma_base:.6f}\n")
    
    best_val_acc = -1.0
    best_params = {'C': SVM_C, 'gamma': SVM_GAMMA}
    
    for c_cand in C_CANDIDATES:
        for g_mult in GAMMA_MULTIPLIERS:
            gamma_val = gamma_base * g_mult
            cand_svc = SVC(C=c_cand, gamma=gamma_val, kernel=KERNEL, cache_size=2048, random_state=RANDOM_STATE)
            cand_svc.fit(X_tr_sub_scaled, y_tr_sub)
            val_acc = cand_svc.score(X_va_sub_scaled, y_va_sub)
            print(f"C={c_cand:4.1f} | gamma={g_mult:4.2f}x scale ({gamma_val:.6f}) -> Disjoint Val Acc: {val_acc*100:.2f}%")
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_params = {'C': c_cand, 'gamma': gamma_val, 'gamma_mult': g_mult}
                
    print("-" * 68)
    print(f"[HASIL TUNING] Parameter terbaik terpilih berdasarkan Disjoint Validation Set:")
    print(f"  C = {best_params['C']}, gamma = {best_params['gamma']} (Val Acc: {best_val_acc*100:.2f}%)")
    SVM_C = best_params['C']
    SVM_GAMMA = best_params['gamma']

print()
print(f"2. Melatih Support Vector Machine Final pada seluruh 50.000 sampel (RBF Kernel, C={SVM_C}, gamma='{SVM_GAMMA}')...")
svm_model = SVC(
    C=SVM_C,
    gamma=SVM_GAMMA,
    kernel=KERNEL,
    cache_size=4096,
    random_state=RANDOM_STATE,
    decision_function_shape='ovr'
)
svm_model.fit(X_train_scaled, y_train)
print("[BERHASIL] SVM Classifier telah selesai dilatih!")

sample_acc = accuracy_score(y_train[:5000], svm_model.predict(X_train_scaled[:5000]))
print(f"Akurasi SVM pada sampel training: {sample_acc * 100:.2f}%")


Menstandarisasi vektor fitur training dengan StandardScaler...
[BERHASIL] Fitur training telah distandarisasi. Shape: (50000, 512)

2. Melatih Support Vector Machine Final pada seluruh 50.000 sampel (RBF Kernel, C=10.0, gamma='scale')...
[BERHASIL] SVM Classifier telah selesai dilatih!
Akurasi SVM pada sampel training: 99.38%


## 📦 2. Menyimpan Scaler & Model SVM ke File Model .keras Tunggal


In [3]:
cnn_model_path = os.path.join(ARTIFACT_DIR, 'cnn_model.keras')
cnn_model = tf.keras.models.load_model(cnn_model_path)
final_keras_path = os.path.join(ARTIFACT_DIR, 'custom_cnn_cifar10_final.keras')

# 1. Simpan standalone scaler.pkl dan svm_model.pkl untuk kepastian kompatibilitas
scaler_pkl_path = os.path.join(ARTIFACT_DIR, 'scaler.pkl')
svm_pkl_path = os.path.join(ARTIFACT_DIR, 'svm_model.pkl')
with open(scaler_pkl_path, 'wb') as f:
    pickle.dump(scaler, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(svm_pkl_path, 'wb') as f:
    pickle.dump(svm_model, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"[BERHASIL] Scaler dan SVM Model disimpan ke standalone .pkl: {scaler_pkl_path} & {svm_pkl_path}")

# 2. Simpan dan injeksikan ke file .keras tunggal
metadata = {
    'method': 'Grayscale NTSC CNN + SVM',
    'feature': 'Grayscale NTSC',
    'svm_C': float(SVM_C),
    'svm_gamma': str(SVM_GAMMA),
    'feature_dim': int(X_train_features.shape[1])
}

save_and_inject_final_model(cnn_model, scaler, svm_model, final_keras_path, metadata)

v_cnn, v_scaler, v_svm, v_meta = load_injected_final_model(final_keras_path)
print(f"[VERIFIKASI] Model final .keras berhasil memuat CNN ({v_cnn.name}), Scaler ({v_scaler}), dan SVM ({v_svm})!")


[BERHASIL] Scaler dan SVM Model disimpan ke standalone .pkl: cifar10_artifacts/grayscale_ntsc\scaler.pkl & cifar10_artifacts/grayscale_ntsc\svm_model.pkl
[BERHASIL] Seluruh model (CNN + Scaler + SVM) tersimpan dalam SATU file (.keras): cifar10_artifacts/grayscale_ntsc\custom_cnn_cifar10_final.keras
[VERIFIKASI] Model final .keras berhasil memuat CNN (custom_cnn_ntsc), Scaler (StandardScaler()), dan SVM (SVC(C=10.0, cache_size=4096, random_state=23092026))!
